<a href="https://colab.research.google.com/github/anthony-yt/5g-rural-coverage-heuristics/blob/anthony-ver/analytics/analisis_sensibilidad_parametros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Parámetros del experimento

In [1]:
using Printf
using Statistics

LAMBDA_EQUIPO = 0.5
FACTORES = [0.0, 0.125, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0]
SEMILLAS = 1:10
ITER = 2000000
REPO = "https://github.com/anthony-yt/5g-rural-coverage-heuristics"
RAMA = "diego-task-1.1"

run(`git clone -b $RAMA $REPO.git`)

[ Info: Precompiling Statistics [10745b16-79ce-11e8-11f9-7d13ad32a3b2](cache misses: wrong source (1), target mismatch (1), mismatched flags (1))
[ Info: Precompiling Statistics [10745b16-79ce-11e8-11f9-7d13ad32a3b2] (cache misses: wrong source (2), target mismatch (2), mismatched flags (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
Cloning into '5g-rural-coverage-heuristics'...


Process(`git clone -b diego-task-1.1 https://github.com/anthony-yt/5g-rural-coverage-heuristics.git`, ProcessExited(0))

## Parámetros del modelo

In [2]:
const NLADO = 10
const NANT = NLADO * NLADO
const FREC = 3500.0
const PMAX = 1.0
const UMBRAL = -80.0
const DZ = (30.0 - 1.5) / 1000.0
const PERDIDA_BASE = 32.4 + 20.0 * log10(FREC)
const K = 10.0^((UMBRAL - 30.0 + PERDIDA_BASE) / 10.0)
const DVECINA = 1.5

const PASO_INICIAL = 0.20
const PASO_FINAL = 0.02
const ACEPTACION = 0.5
const ENFRIAMIENTO = 0.001

0.001

## Generador Congruencial Mixto

In [3]:
mutable struct Generador
    X::UInt64
end

function Uniforme(G)
    G.X = G.X * 0x5851f42d4c957f2d + 0x14057b7ef767814f
    return (G.X >> 11) / 9007199254740992.0
end

function Entero(G, N)
    return floor(Int, Uniforme(G) * N) + 1
end

function Normal(G)
    return sqrt(-2 * log(1 - Uniforme(G))) * cos(2 * pi * Uniforme(G))
end

function NuevoGenerador(SEMILLA)
    G = Generador(UInt64(SEMILLA))
    for _ in 1:10
        Uniforme(G)
    end
    return G
end

NuevoGenerador (generic function with 1 method)

## Modelo de la red

In [4]:
struct Modelo
    AX::Vector{Float64}
    AY::Vector{Float64}
    PX::Vector{Float64}
    PY::Vector{Float64}
    ALCANCE::Vector{Vector{Int}}
    MINIMA::Vector{Vector{Float64}}
    VECINAS::Vector{Vector{Int}}
    PESOS::Vector{Vector{Float64}}
    SENALES::Vector{Int}
    IMAX::Vector{Float64}
end

function CrearModelo()
    AX = Float64[]
    AY = Float64[]
    for x in 0.5:1.0:9.5, y in 0.5:1.0:9.5
        push!(AX, x)
        push!(AY, y)
    end
    PX = Float64[]
    PY = Float64[]
    for x in 0.1:0.2:9.9, y in 0.1:0.2:9.9
        push!(PX, x)
        push!(PY, y)
    end
    ALCANCE = Vector{Int}[]
    MINIMA = Vector{Float64}[]
    for I in 1:NANT
        PARES = Tuple{Float64,Int}[]
        for J in eachindex(PX)
            PM = ((PX[J] - AX[I])^2 + (PY[J] - AY[I])^2 + DZ^2) * K
            if PM <= PMAX
                push!(PARES, (PM, J))
            end
        end
        sort!(PARES)
        push!(ALCANCE, [P[2] for P in PARES])
        push!(MINIMA, [P[1] for P in PARES])
    end
    VECINAS = [Int[] for _ in 1:NANT]
    PESOS = [Float64[] for _ in 1:NANT]
    for I in 1:NANT, J in I+1:NANT
        D = sqrt((AX[I] - AX[J])^2 + (AY[I] - AY[J])^2)
        if D <= DVECINA
            push!(VECINAS[I], J)
            push!(PESOS[I], 1.0 / D^2)
            push!(VECINAS[J], I)
            push!(PESOS[J], 1.0 / D^2)
        end
    end
    M = Modelo(AX, AY, PX, PY, ALCANCE, MINIMA, VECINAS, PESOS, zeros(Int, length(PX)), [1.0, 1.0])
    X = fill(PMAX, NANT)
    ContarSenales(M, X)
    M.IMAX[1] = Mutua(M, X)
    M.IMAX[2] = Sobrantes(M)
    return M
end

CrearModelo (generic function with 1 method)

## Función objetivo

In [5]:
function ContarSenales(M, X)
    fill!(M.SENALES, 0)
    for I in 1:NANT
        for J in eachindex(M.ALCANCE[I])
            if M.MINIMA[I][J] > X[I]
                break
            end
            M.SENALES[M.ALCANCE[I][J]] += 1
        end
    end
end

function Mutua(M, X)
    S = 0.0
    for I in 1:NANT
        for J in eachindex(M.VECINAS[I])
            if M.VECINAS[I][J] > I
                S += M.PESOS[I][J] * X[I] * X[M.VECINAS[I][J]]
            end
        end
    end
    return S
end

function Sobrantes(M)
    return sum(max(S - 1, 0) for S in M.SENALES)
end

function Evaluar(M, X)
    ContarSenales(M, X)
    COB = count(S -> S >= 1, M.SENALES) / length(M.SENALES)
    INT = 0.5 * (Mutua(M, X) / M.IMAX[1]) + 0.5 * (Sobrantes(M) / M.IMAX[2])
    return COB, INT
end

function Fitness(M, X, LAMBDA)
    COB, INT = Evaluar(M, X)
    return COB - LAMBDA * INT
end

Fitness (generic function with 1 method)

## Recocido simulado

In [6]:
function Cambio(M, X, I, PNUEVA, LAMBDA)
    PTS = M.ALCANCE[I]
    J0 = searchsortedlast(M.MINIMA[I], X[I])
    J1 = searchsortedlast(M.MINIMA[I], PNUEVA)
    DCUB = 0
    DSOB = 0
    for J in J0+1:J1
        if M.SENALES[PTS[J]] == 0
            DCUB += 1
        else
            DSOB += 1
        end
    end
    for J in J1+1:J0
        if M.SENALES[PTS[J]] == 1
            DCUB -= 1
        else
            DSOB -= 1
        end
    end
    S = 0.0
    for J in eachindex(M.VECINAS[I])
        S += M.PESOS[I][J] * X[M.VECINAS[I][J]]
    end
    DINT = 0.5 * ((PNUEVA - X[I]) * S) / M.IMAX[1] + 0.5 * DSOB / M.IMAX[2]
    return DCUB / length(M.SENALES) - LAMBDA * DINT, J0, J1
end

function Aplicar(M, X, I, PNUEVA, J0, J1)
    for J in J0+1:J1
        M.SENALES[M.ALCANCE[I][J]] += 1
    end
    for J in J1+1:J0
        M.SENALES[M.ALCANCE[I][J]] -= 1
    end
    X[I] = PNUEVA
end

function PotenciaNecesaria(M, X)
    Z = zeros(NANT)
    for I in 1:NANT
        J = searchsortedlast(M.MINIMA[I], X[I])
        if J > 0
            Z[I] = M.MINIMA[I][J]
        end
    end
    return Z
end

function TemperaturaInicial(M, X, LAMBDA, G)
    S = 0.0
    N = 0
    for _ in 1:1000
        I = Entero(G, NANT)
        D, J0, J1 = Cambio(M, X, I, clamp(X[I] + PASO_INICIAL * PMAX * Normal(G), 0.0, PMAX), LAMBDA)
        if D < 0
            S -= D
            N += 1
        end
    end
    return N == 0 ? 0.001 : (S / N) / (-log(ACEPTACION))
end

function RecocidoSimulado(M, LAMBDA, SEMILLA, ITER)
    G = NuevoGenerador(SEMILLA)
    X = [Uniforme(G) * PMAX for _ in 1:NANT]
    F = Fitness(M, X, LAMBDA)
    MEJOR = copy(X)
    FMEJOR = F
    T = TemperaturaInicial(M, X, LAMBDA, G)
    ALFA = ENFRIAMIENTO^(1 / ITER)
    for IT in 1:ITER
        PASO = PASO_INICIAL + (PASO_FINAL - PASO_INICIAL) * IT / ITER
        I = Entero(G, NANT)
        PNUEVA = clamp(X[I] + PASO * PMAX * Normal(G), 0.0, PMAX)
        D, J0, J1 = Cambio(M, X, I, PNUEVA, LAMBDA)
        if D >= 0 || Uniforme(G) < exp(D / T)
            Aplicar(M, X, I, PNUEVA, J0, J1)
            F += D
            if F > FMEJOR + 1e-9
                FMEJOR = F
                copyto!(MEJOR, X)
            end
        end
        T *= ALFA
    end
    MEJOR = PotenciaNecesaria(M, MEJOR)
    return MEJOR, Fitness(M, MEJOR, LAMBDA)
end

RecocidoSimulado (generic function with 1 method)

## Comprobaciones

In [7]:
function ObtenerRepositorio()
    if isfile("../core/fitness.jl")
        return true
    end
    try
        if !isdir("5g-rural-coverage-heuristics")
            run(`git clone --quiet --depth 1 --branch $RAMA $REPO`)
        end
        cd("5g-rural-coverage-heuristics/analytics")
        return isfile("../core/fitness.jl")
    catch
        println("No se pudo clonar el repositorio: se omite la comparacion con fitness().")
        return false
    end
end

function Comprobar(M, LAMBDA)
    println("COMPROBACIONES")
    println("==============")
    for (NOMBRE, X) in (("Todas apagadas", zeros(NANT)), ("Todas a 1 W", fill(PMAX, NANT)), ("Todas a 0.35 W", fill(0.35, NANT)))
        COB, INT = Evaluar(M, X)
        @printf("  %-15s cobertura %6.2f %%   interferencia %.4f   fitness %.4f\n", NOMBRE, 100 * COB, INT, COB - LAMBDA * INT)
    end
    G = NuevoGenerador(12345)
    X = [Uniforme(G) * PMAX for _ in 1:NANT]
    F = Fitness(M, X, LAMBDA)
    DIF = 0.0
    for _ in 1:2000
        I = Entero(G, NANT)
        PNUEVA = Uniforme(G) < 0.2 ? 0.0 : Uniforme(G) * PMAX
        D, J0, J1 = Cambio(M, X, I, PNUEVA, LAMBDA)
        Aplicar(M, X, I, PNUEVA, J0, J1)
        F += D
        DIF = max(DIF, abs(F - Fitness(M, X, LAMBDA)))
    end
    @printf("  Fitness por partes contra completo (2000 cambios): diferencia maxima %.1e\n", DIF)
    if isdefined(Main, :Rural5GFitness)
        DIF = 0.0
        for k in 1:100
            X = [Uniforme(G) < 0.3 ? 0.0 : Uniforme(G) * PMAX for _ in 1:NANT]
            if iseven(k)
                X = PotenciaNecesaria(M, X)
            end
            DIF = max(DIF, abs(Fitness(M, X, LAMBDA) - Main.Rural5GFitness.fitness(X; lambda = LAMBDA)))
        end
        @printf("  Contra fitness() de core/fitness.jl (100 vectores): diferencia maxima %.1e\n", DIF)
        if DIF > 1e-9
            println("  ATENCION: ya no coincide con core/fitness.jl, hay que actualizar este codigo.")
        end
    end
    println()
end

Comprobar (generic function with 1 method)

## Análisis de sensibilidad

In [8]:
function Analizar(M, LAMBDA)
    FS = Float64[]
    COBS = Float64[]
    INTS = Float64[]
    ENCS = Float64[]
    POTS = Float64[]
    MEJOR = zeros(NANT)
    for SEMILLA in SEMILLAS
        X, F = RecocidoSimulado(M, LAMBDA, SEMILLA, ITER)
        COB, INT = Evaluar(M, X)
        if isempty(FS) || F > maximum(FS)
            MEJOR = X
        end
        push!(FS, F)
        push!(COBS, 100 * COB)
        push!(INTS, INT)
        push!(ENCS, count(P -> P > 0, X))
        push!(POTS, sum(X))
    end
    return (LAMBDA = LAMBDA, FS = FS, COBS = COBS, INTS = INTS, ENC = mean(ENCS), POT = mean(POTS),
            FMAX = Fitness(M, fill(PMAX, NANT), LAMBDA), MEJOR = MEJOR)
end

Analizar (generic function with 1 method)

## Salidas

In [9]:
function ImprimirTabla(V)
    println("-------------------------------------------------------------------------------------------")
    println(" LAMBDA    FITNESS (media +- desv)    MEJOR    COBERTURA %    INTERFERENCIA    ENCENDIDAS    AL MAXIMO")
    println("-------------------------------------------------------------------------------------------")
    for R in V
        @printf(" %6s      %7.4f +- %6.4f    %7.4f      %6.2f         %6.4f          %5.1f      %8.4f\n",
                R.LAMBDA, mean(R.FS), std(R.FS), maximum(R.FS), mean(R.COBS), mean(R.INTS), R.ENC, R.FMAX)
    end
    println("-------------------------------------------------------------------------------------------")
end

function ImprimirHTML(V, FILE, title)
    FF = open(FILE, "w")
    write(FF, """
        <html>
        <h1>$title</h1>
        <hr>
        <p>Cada fila es la media de $(length(SEMILLAS)) corridas
        de recocido simulado ($ITER iteraciones).</p>
        <table cellpadding=7 cellspacing=0 border=1>
        <tr bgcolor=#C0C0C0>
        <td>LAMBDA</td>
        <td>Fitness</td>
        <td>Mejor</td>
        <td>Cobertura %</td>
        <td>Interferencia</td>
        <td>Antenas encendidas</td>
        <td>Potencia total (W)</td>
        <td>Todas al maximo</td>
        </tr>""")
    for R in V
        if R.LAMBDA == LAMBDA_EQUIPO
            C1 = C2 = "#FFFF00"
        else
            C1 = "#AFEEEE"
            C2 = "#FFF8DC"
        end
        write(FF, @sprintf("""
        <tr>
        <td bgcolor=%s>%s</td>
        <td bgcolor=%s>%.4f +- %.4f</td>
        <td bgcolor=%s>%.4f</td>
        <td bgcolor=%s>%.2f +- %.2f</td>
        <td bgcolor=%s>%.4f</td>
        <td bgcolor=%s>%.1f</td>
        <td bgcolor=%s>%.1f</td>
        <td bgcolor=%s>%.4f</td>
        </tr>""", C1, R.LAMBDA, C2, mean(R.FS), std(R.FS), C2, maximum(R.FS), C2, mean(R.COBS), std(R.COBS),
                  C2, mean(R.INTS), C2, R.ENC, C2, R.POT, C2, R.FMAX))
    end
    write(FF, """</table>
        </html>""")
    close(FF)
end

function GuardarCSV(V, FILE)
    FF = open(FILE, "w")
    write(FF, "lambda,fitness_medio,fitness_desv,fitness_mejor,cobertura_pct,interferencia,antenas_encendidas,potencia_total_w,fitness_todas_al_maximo\n")
    for R in V
        write(FF, @sprintf("%s,%.5f,%.5f,%.5f,%.3f,%.5f,%.1f,%.3f,%.5f\n", R.LAMBDA, mean(R.FS), std(R.FS), maximum(R.FS),
                           mean(R.COBS), mean(R.INTS), R.ENC, R.POT, R.FMAX))
    end
    close(FF)
end

function GuardarPotencias(M, R, FILE)
    FF = open(FILE, "w")
    write(FF, "ID_Antena,Pos_X,Pos_Y,Potencia_W\n")
    for I in 1:NANT
        write(FF, string(I, ",", M.AX[I], ",", M.AY[I], ",", R.MEJOR[I], "\n"))
    end
    close(FF)
end

GuardarPotencias (generic function with 1 method)

## Módulo principal

In [10]:
if ObtenerRepositorio()
    include(abspath("../core/fitness.jl"))
end

M = CrearModelo()
Comprobar(M, LAMBDA_EQUIPO)

V = []
for FACTOR in FACTORES
    LAMBDA = LAMBDA_EQUIPO * FACTOR
    R = Analizar(M, LAMBDA)
    push!(V, R)
end

mkpath("output")
ImprimirHTML(V, "output/sensibilidad_lambda.html", "Sensibilidad del peso LAMBDA")
GuardarCSV(V, "output/sensibilidad_lambda.csv")
GuardarPotencias(M, V[findfirst(R -> R.LAMBDA == LAMBDA_EQUIPO, V)], "output/potencias_lambda_equipo.csv")

html_path = abspath("output/sensibilidad_lambda.html")
csv_path = abspath("output/sensibilidad_lambda.csv")
potencias_path = abspath("output/potencias_lambda_equipo.csv")

run(`zip -jo /content/resultados_sensibilidad.zip $html_path $csv_path $potencias_path`)

COMPROBACIONES
  Todas apagadas  cobertura   0.00 %   interferencia 0.0000   fitness 0.0000
  Todas a 1 W     cobertura 100.00 %   interferencia 1.0000   fitness 0.5000
  Todas a 0.35 W  cobertura 100.00 %   interferencia 0.2346   fitness 0.8827
  Fitness por partes contra completo (2000 cambios): diferencia maxima 1.2e-15
  Contra fitness() de core/fitness.jl (100 vectores): diferencia maxima 0.0e+00

  adding: sensibilidad_lambda.html (deflated 80%)
  adding: sensibilidad_lambda.csv (deflated 53%)
  adding: potencias_lambda_equipo.csv (deflated 69%)


Process(`zip -jo /content/resultados_sensibilidad.zip /content/5g-rural-coverage-heuristics/analytics/output/sensibilidad_lambda.html /content/5g-rural-coverage-heuristics/analytics/output/sensibilidad_lambda.csv /content/5g-rural-coverage-heuristics/analytics/output/potencias_lambda_equipo.csv`, ProcessExited(0))